# Tabular Data Cleaning
See *patient_data_instructions.md* for dataset info

In [2]:
import pandas as pd
import findspark
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("initial_data_cleaning").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/22 18:15:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Cleaning/Combining datasets based on datatype

In [33]:
from functools import reduce
from pyspark.sql import DataFrame

### Datatype: Demographic Data (DEMO)

In [3]:
demo_l_location = "hdfs://localhost:9000/user/jj/final_proj/csv_files/DEMO_L.csv"

demo_l = spark.read.csv(demo_l_location, inferSchema = True, header = True)

In [34]:
# Locations
demo_l_location = "hdfs://localhost:9000/user/jj/final_proj/csv_files/DEMO_L.csv"
demo_p_location = "hdfs://localhost:9000/user/jj/final_proj/csv_files/P_DEMO.csv"
demo_i_location = "hdfs://localhost:9000/user/jj/final_proj/csv_files/DEMO_I.csv"
demo_h_location = "hdfs://localhost:9000/user/jj/final_proj/csv_files/DEMO_H.csv"

# Readings
demo_l = spark.read.csv(demo_l_location, inferSchema = True, header = True)
demo_p = spark.read.csv(demo_p_location, inferSchema = True, header = True)
demo_i = spark.read.csv(demo_i_location, inferSchema = True, header = True)
demo_h = spark.read.csv(demo_h_location, inferSchema = True, header = True)

# Place into array
demos = [demo_l, demo_p, demo_i, demo_h]

In [35]:
# Selecting only important columns
demo_cols = ['SEQN', 'RIAGENDR', 'RIDAGEYR', 'RIDRETH3']
demo_cols_renamed = ['SEQN', 'Gender', 'Age', 'Race']

for i in range(len(demos)):
    demos[i] = demos[i].select(demo_cols).toDF(*demo_cols_renamed)

In [45]:
demo_l = demos[0]
demo_p = demos[1]
demo_i = demos[2]
demo_h = demos[3]

In [46]:
demo_l.describe().show()

+-------+----------------+-------------------+--------------------+------------------+
|summary|            SEQN|             Gender|                 Age|              Race|
+-------+----------------+-------------------+--------------------+------------------+
|  count|           11933|              11933|               11933|             11933|
|   mean|        136344.0| 1.5328081789994135|  38.317858040727394|3.3205396798793263|
| stddev|3444.90471566341|0.49894336877363876|   25.60199011389116|1.5183792913880416|
|    min|        130378.0|                1.0|5.397605346934028...|               1.0|
|    max|        142310.0|                2.0|                80.0|               7.0|
+-------+----------------+-------------------+--------------------+------------------+



In [43]:
demo = reduce(DataFrame.unionByName, demos)
demo.describe().show()

[Stage 58:=============================>                            (2 + 2) / 4]

+-------+------------------+------------------+------------------+-----------------+
|summary|              SEQN|            Gender|               Age|             Race|
+-------+------------------+------------------+------------------+-----------------+
|  count|             47639|             47639|             47639|            47639|
|   mean|107747.31121560067|1.5131929721446713| 34.02044543336342|3.345578202733055|
| stddev|22425.859947651064|0.4998311612371023|25.225247534762826|1.610803520358708|
|    min|           73557.0|               1.0|               0.0|              1.0|
|    max|          142310.0|               2.0|              80.0|              7.0|
+-------+------------------+------------------+------------------+-----------------+



## EDA

### Datatype: Demographic Data (DEMO)